# Transfer Learning (PyTorch)
In this notebook, you will perform transfer learning to train CIFAR-10 dataset on the ResNet50 model available in torchvision.

> This notebook is a PyTorch port of the original TensorFlow/Keras lab. `tf.keras.applications.ResNet50` becomes `torchvision.models.resnet50`, `preprocess_input` becomes the standard ImageNet mean/std normalization, and `model.fit` becomes an explicit training loop.

## Imports

In [ ]:
import os, re, time, json
import PIL.Image, PIL.ImageFont, PIL.ImageDraw
import numpy as np
from matplotlib import pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import CIFAR10
from torchvision.models import resnet50, ResNet50_Weights
from torchinfo import summary

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

## Parameters

- Define the batch size
- Define the class (category) names

In [ ]:
BATCH_SIZE = 32
classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

Define some functions that will help you to create some visualizations. (These will be used later)

In [ ]:
#@title Visualization Utilities[RUN ME]
#Matplotlib config
plt.rc('image', cmap='gray')
plt.rc('grid', linewidth=0)
plt.rc('xtick', top=False, bottom=False, labelsize='large')
plt.rc('ytick', left=False, right=False, labelsize='large')
plt.rc('axes', facecolor='F8F8F8', titlesize="large", edgecolor='white')
plt.rc('text', color='a8151a')
plt.rc('figure', facecolor='F0F0F0')# Matplotlib fonts
MATPLOTLIB_FONT_DIR = os.path.join(os.path.dirname(plt.__file__), "mpl-data/fonts/ttf")
# utility to display a row of digits with their predictions
def display_images(digits, predictions, labels, title):

  '''
  Displays 10 random images, each labelled with the class the model predicted.

  Args:
    digits (array) -- images, shape (N, 32, 32, 3)
    predictions (array) -- predicted class ids
    labels (array) -- ground truth class ids
    title (string) -- title for the figure
  '''
  n = 10

  indexes = np.random.choice(len(predictions), size=n)
  n_digits = digits[indexes]
  n_predictions = predictions[indexes]
  n_predictions = n_predictions.reshape((n,))
  n_labels = labels[indexes]

  fig = plt.figure(figsize=(20, 4))
  plt.title(title)
  plt.yticks([])
  plt.xticks([])

  for i in range(10):
    ax = fig.add_subplot(1, 10, i+1)
    class_index = n_predictions[i]

    plt.xlabel(classes[class_index])
    plt.xticks([])
    plt.yticks([])
    plt.imshow(n_digits[i])

# utility to display training and validation curves
def plot_metrics(history, metric_name, title, ylim=5):
  '''
  Plots a training metric and its validation counterpart against the epoch number.

  Args:
    history (dict) -- metric name to list of per-epoch values
    metric_name (string) -- key to plot, for example 'loss'
    title (string) -- title for the figure
    ylim (float) -- upper limit of the y axis
  '''
  plt.title(title)
  plt.ylim(0,ylim)
  plt.plot(history[metric_name],color='blue',label=metric_name)
  plt.plot(history['val_' + metric_name],color='green',label='val_' + metric_name)

## Loading and Preprocessing Data
[CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) dataset has 32 x 32 RGB images belonging to 10 classes. You will load the dataset from torchvision.

In [ ]:
train_set = CIFAR10(root="data", train=True, download=True)
valid_set = CIFAR10(root="data", train=False, download=True)

# torchvision stores the images as a (N, 32, 32, 3) uint8 numpy array and the labels as a list
training_images, training_labels = train_set.data, np.array(train_set.targets)
validation_images, validation_labels = valid_set.data, np.array(valid_set.targets)

### Visualize Dataset

Use the `display_image` to view some of the images and their class labels.

In [ ]:
display_images(training_images, training_labels, training_labels, "Training Data" )

In [ ]:
display_images(validation_images, validation_labels, validation_labels, "Training Data" )

In [ ]:
validation_images[0].astype('float32').shape

### Preprocess Dataset
Here, you'll perform normalization on images in training and validation set.
- Keras' `preprocess_input` for ResNet50 does channel-wise mean subtraction. The torchvision ResNet50 weights instead expect inputs scaled to `[0, 1]` and normalized with the ImageNet mean and standard deviation (this is what `ResNet50_Weights.IMAGENET1K_V1.transforms()` does).
- PyTorch uses the channels-first layout, so the images are also transposed from `(N, 32, 32, 3)` to `(N, 3, 32, 32)`.

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype='float32').reshape(1, 3, 1, 1)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype='float32').reshape(1, 3, 1, 1)

def preprocess_image_input(input_images):
  '''
  Scales images to [0, 1], moves the channel axis first, and applies ImageNet normalization.

  Args:
    input_images (array) -- uint8 images, shape (N, 32, 32, 3)

  Returns:
    array -- float32 images, shape (N, 3, 32, 32)
  '''
  input_images = input_images.astype('float32') / 255.0
  input_images = input_images.transpose(0, 3, 1, 2)          # (N, H, W, C) -> (N, C, H, W)
  output_ims = (input_images - IMAGENET_MEAN) / IMAGENET_STD
  return output_ims

In [ ]:
train_X = preprocess_image_input(training_images)
valid_X = preprocess_image_input(validation_images)

## Define the Network
You will be performing transfer learning on **ResNet50** available in torchvision.
- You'll load pre-trained **imagenet weights** to the model.
- You'll choose to retain all layers of **ResNet50** along with the final classification layers.

In [ ]:
'''
Feature Extraction is performed by ResNet50 pretrained on imagenet weights.
Input size is 224 x 224.
'''
def feature_extractor():
  '''
  Builds the ResNet50 backbone that extracts features.

  Returns:
    nn.Sequential -- ResNet50 pretrained on ImageNet, with its pooling and fully connected head removed
  '''
  resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
  # drop the final average pooling and fully connected layers (Keras' include_top=False)
  feature_extractor = nn.Sequential(*list(resnet.children())[:-2])
  return feature_extractor


'''
Defines final dense layers for classification. The last layer outputs the 10 class logits
(the softmax is applied inside nn.CrossEntropyLoss during training and explicitly at prediction time).
'''
def classifier():
  '''
  Builds the dense head that turns ResNet features into class logits.

  Returns:
    nn.Sequential -- head turning (N, 2048, 7, 7) feature maps into 10 class logits
  '''
  return nn.Sequential(
      nn.AdaptiveAvgPool2d(1),                 # GlobalAveragePooling2D
      nn.Flatten(),
      nn.Linear(2048, 1024), nn.ReLU(),
      nn.Linear(1024, 512), nn.ReLU(),
      nn.Linear(512, 10),                      # "classification" layer
  )

'''
Since input image size is (32 x 32), first upsample the image by factor of (7x7) to transform it to (224 x 224)
Connect the feature extraction and "classifier" layers to build the model.
'''
class FinalModel(nn.Module):
  '''
  CIFAR-10 classifier that upsamples 32x32 inputs to 224x224 and runs them through ResNet50.
  '''
  def __init__(self):
    '''
    Builds the upsampler, the ResNet50 feature extractor and the classification head.
    '''
    super().__init__()
    self.resize = nn.Upsample(scale_factor=7)   # nearest-neighbour upsampling, like Keras UpSampling2D
    self.resnet_feature_extractor = feature_extractor()
    self.classification = classifier()

  def forward(self, inputs):
    '''
    Upsamples the small CIFAR image, extracts features, then classifies them.

    Args:
      inputs (tensor) -- batch of images, shape (N, 3, 32, 32)

    Returns:
      tensor -- class logits, shape (N, 10)
    '''
    resize = self.resize(inputs)
    features = self.resnet_feature_extractor(resize)
    classification_output = self.classification(features)
    return classification_output

'''
Define the model and the training configuration.
Use Stochastic Gradient Descent as the optimizer.
Use Cross Entropy (the equivalent of Sparse Categorical CrossEntropy for integer labels) as the loss function.
'''
def define_compile_model(device):
  '''
  Creates the model together with its optimizer and loss function.

  Args:
    device (torch.device) -- device the model is moved to

  Returns:
    (nn.Module, Optimizer, callable) -- the model, its SGD optimizer and the loss function
  '''
  model = FinalModel().to(device)

  optimizer = torch.optim.SGD(model.parameters(), lr=0.01)   # Keras' default SGD learning rate
  loss_fn = nn.CrossEntropyLoss()

  return model, optimizer, loss_fn


model, optimizer, loss_fn = define_compile_model(device)

summary(model, input_size=(1, 3, 32, 32), device=device, depth=2)

## Train the model

In [ ]:
# this will take a while to complete (ResNet50 at 224x224 on 50,000 images per epoch)
EPOCHS = 4

train_loader = DataLoader(TensorDataset(torch.from_numpy(train_X), torch.from_numpy(training_labels).long()), batch_size=64, shuffle=True)
valid_loader = DataLoader(TensorDataset(torch.from_numpy(valid_X), torch.from_numpy(validation_labels).long()), batch_size=64)

history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}


def run_epoch(loader, model, loss_fn, optimizer, device, train, steps=None):
  '''
  Runs one pass over `loader`, training or evaluating.

  Args:
    loader (DataLoader) -- yields (images, labels) batches
    model (nn.Module) -- classifier being trained or evaluated
    loss_fn (callable) -- loss applied to (logits, labels)
    optimizer (Optimizer) -- updates weights; only used when train is True
    device (torch.device) -- device the batches are moved to
    train (bool) -- True updates the weights, False only measures
    steps (int) -- stop after this many batches, or None for the whole loader

  Returns:
    (float, float) -- mean loss and accuracy
  '''
  model.train(train)
  total_loss, correct, count = 0.0, 0, 0
  with torch.set_grad_enabled(train):
    for step, (xb, yb) in enumerate(loader):
      if steps is not None and step >= steps:
        break
      xb, yb = xb.to(device), yb.to(device)
      logits = model(xb)
      loss = loss_fn(logits, yb)
      if train:
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
      total_loss += loss.item() * len(xb)
      correct += (logits.argmax(1) == yb).sum().item()
      count += len(xb)
  return total_loss / count, correct / count


for epoch in range(EPOCHS):
  start = time.time()
  train_loss, train_acc = run_epoch(train_loader, model, loss_fn, optimizer, device, train=True)
  val_loss, val_acc = run_epoch(valid_loader, model, loss_fn, optimizer, device, train=False)
  history['loss'].append(train_loss); history['accuracy'].append(train_acc)
  history['val_loss'].append(val_loss); history['val_accuracy'].append(val_acc)
  print(f"Epoch {epoch + 1}/{EPOCHS} - {time.time() - start:.0f}s - loss: {train_loss:.4f} - accuracy: {train_acc:.4f} "
        f"- val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")

## Evaluate the Model

Calculate the loss and accuracy metrics on the validation set.

In [ ]:
loss, accuracy = run_epoch(valid_loader, model, loss_fn, optimizer, device, train=False)
print(f"loss: {loss:.4f} - accuracy: {accuracy:.4f}")

### Plot Loss and Accuracy Curves

Plot the loss (in blue) and validation loss (in green).

In [ ]:
plot_metrics(history, "loss", "Loss")

Plot the training accuracy (blue) as well as the validation accuracy (green).

In [ ]:
plot_metrics(history, "accuracy", "Accuracy")

### Visualize predictions
You can take a look at the predictions on the validation set.

In [ ]:
model.eval()
probabilities = []
with torch.no_grad():
  for xb, _ in valid_loader:
    probabilities.append(torch.softmax(model(xb.to(device)), dim=1).cpu())
probabilities = torch.cat(probabilities).numpy()
probabilities = np.argmax(probabilities, axis = 1)

display_images(validation_images, probabilities, validation_labels, "Bad predictions indicated in red.")